In [6]:
import numpy as np
import pandas as pd
import sys
sys.path.append("../backend")
from urllib.parse import urlparse

from feature_extraction import extract_features_batch

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GroupKFold, cross_val_score
from sklearn.metrics import (
    confusion_matrix,
    accuracy_score,
    classification_report,
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score
)
from sklearn.dummy import DummyClassifier

from xgboost import XGBClassifier
import joblib


def get_domain(url):
    if "://" not in url:
        url = "http://" + url
    netloc = urlparse(url).netloc
    if netloc.startswith("www."):
        netloc = netloc[4:]
    return netloc


dataset = pd.read_csv("url_features_extracted1 (1).csv")

dataset = dataset.dropna(subset=["ClassLabel"])
dataset["ClassLabel"] = dataset["ClassLabel"].astype(int)

print("Duplicate URLs:", dataset["URL"].duplicated().sum())

conflicting = dataset.groupby("URL")["ClassLabel"].nunique()
print("URLs with conflicting labels:", (conflicting > 1).sum())

dataset["Domain"] = dataset["URL"].apply(get_domain)

feature_rows = extract_features_batch(dataset["URL"])
X = pd.DataFrame(feature_rows)
numeric_cols = list(X.columns)

y = dataset["ClassLabel"]
groups = dataset["Domain"]

preprocessor = ColumnTransformer([
    ("num", StandardScaler(), numeric_cols)
])

pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", XGBClassifier(
        random_state=42,
        eval_metric="logloss"
    ))
])

gkf = GroupKFold(n_splits=5)

train_idx, test_idx = next(gkf.split(X, y, groups))

X_train = X.iloc[train_idx]
X_test = X.iloc[test_idx]

y_train = y.iloc[train_idx]
y_test = y.iloc[test_idx]

pipeline.fit(X_train, y_train)

y_pred = pipeline.predict(X_test)
y_prob = pipeline.predict_proba(X_test)[:, 1]

baseline = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", DummyClassifier(strategy="most_frequent"))
])

baseline.fit(X_train, y_train)
baseline_pred = baseline.predict(X_test)

print("\n--- Baseline ---")
print("Accuracy: {:.2f}%".format(
    accuracy_score(y_test, baseline_pred) * 100
))

print("\n--- XGBoost Results ---")

cm = confusion_matrix(y_test, y_pred)

print("\nConfusion Matrix")
print(cm)

accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
roc_auc = roc_auc_score(y_test, y_prob)

tn, fp, fn, tp = cm.ravel()

false_positive_rate = fp / (fp + tn)
false_negative_rate = fn / (fn + tp)

print("\nAccuracy: {:.2f}%".format(
    accuracy * 100
))

print("\nClassification Report")
print(classification_report(
    y_test,
    y_pred,
    target_names=["Phishing", "Legitimate"]
))

print("\nPerformance Summary")
print(f"Precision:            {precision:.4f}")
print(f"Recall:               {recall:.4f}")
print(f"F1 Score:             {f1:.4f}")
print(f"ROC-AUC:              {roc_auc:.4f}")
print(f"False Positive Rate:  {false_positive_rate:.2%}")
print(f"False Negative Rate:  {false_negative_rate:.2%}")

cv_scores = cross_val_score(
    pipeline,
    X,
    y,
    groups=groups,
    cv=gkf,
    scoring="accuracy"
)

print("\n--- GroupKFold Cross Validation ---")
print("Mean Accuracy: {:.2f}%".format(cv_scores.mean() * 100))
print("Std Dev: {:.2f}%".format(cv_scores.std() * 100))

pipeline.fit(X, y)

importance = pd.Series(
    pipeline.named_steps["classifier"].feature_importances_,
    index=numeric_cols
).sort_values(ascending=False)

print("\n--- Feature Importance ---")
print(importance)

joblib.dump(pipeline, "phishing_pipeline.pkl")
print("\nModel saved as phishing_pipeline.pkl")

Duplicate URLs: 346
URLs with conflicting labels: 0

--- Baseline ---
Accuracy: 62.60%

--- XGBoost Results ---

Confusion Matrix
[[12624    49]
 [   29  7542]]

Accuracy: 99.61%

Classification Report
              precision    recall  f1-score   support

    Phishing       1.00      1.00      1.00     12673
  Legitimate       0.99      1.00      0.99      7571

    accuracy                           1.00     20244
   macro avg       1.00      1.00      1.00     20244
weighted avg       1.00      1.00      1.00     20244


Performance Summary
Precision:            0.9935
Recall:               0.9962
F1 Score:             0.9949
ROC-AUC:              1.0000
False Positive Rate:  0.39%
False Negative Rate:  0.38%

--- GroupKFold Cross Validation ---
Mean Accuracy: 99.76%
Std Dev: 0.12%

--- Feature Importance ---
https_flag                   0.688032
token_count                  0.249127
suspicious_file_extension    0.017027
percentage_numeric_chars     0.015666
subdomain_count         

I originally thought that there was more data leakage problems, specifically I thought that XGBoost was just finding out the patterns from when the data was just being collected so I made it so that no domain in the test set ever appeared in the training phase. This is the model that I am going with for the project. Refactored for new dataset because I was unhappy with the results the previous model produced.